In [ ]:
# maps_and_plots.py
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

from core_glaucogate import ConvNeXtProGlaucoGate
from step6_domain_generalization_boost import DomainInvariantFundusDataset

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 300
})

def denorm_img(tensor_chw):
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    img = tensor_chw.cpu().numpy() * std + mean
    return np.clip(np.transpose(img, (1, 2, 0)) * 255.0, 0, 255).astype(np.uint8)

def apply_colormap_overlay(base_rgb, map_2d, alpha=0.45, cmap=cv2.COLORMAP_JET):
    h, w, _ = base_rgb.shape
    m_norm = (map_2d - map_2d.min()) / (map_2d.max() - map_2d.min() + 1e-6)
    m_uint8 = np.uint8(255 * m_norm)
    m_resized = cv2.resize(m_uint8, (w, h), interpolation=cv2.INTER_CUBIC)
    heat_bgr = cv2.applyColorMap(m_resized, cmap)
    heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(base_rgb, 1.0 - alpha, heat_rgb, alpha, 0)

def generate_multidomain_heatmaps(ckpt_path, out_dir="./results/figures"):
    os.makedirs(out_dir, exist_ok=True)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = ConvNeXtProGlaucoGate(
        backbone_name="convnext_base.fb_in22k_ft_in1k", pretrained=False, fusion_mode="multiplicative"
    ).to(device)
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device)["model_state"], strict=False)
    model.eval()

    datasets_info = [
        ("AIROGS (Test)", "./data_splits/airogs_splits.csv", "test"),
        ("PAPILA (External)", "./data_splits/papila_external.csv", None),
        ("Chaksu (External)", "./data_splits/chaksu_external.csv", None)
    ]

    fig, axes = plt.subplots(3, 6, figsize=(20, 10.5))
    col_titles = [
        "(a) Fundus + STN ONH Box",
        "(b) High-Res Local ONH Crop",
        "(c) Quality Map $M_Q(x,y)$",
        "(d) Q-CAF Gate $G(x,y)$",
        "(e) Polar-ISNT Saliency",
        "(f) GlaucoGate Diagnostic Overlay"
    ]

    for row_idx, (dname, csv_path, split_name) in enumerate(datasets_info):
        if not os.path.exists(csv_path):
            continue
        ds = DomainInvariantFundusDataset(csv_path, split=split_name, is_train=False)
        pos_indices = np.where(ds.df["label"].values == 1)[0]
        chosen_idx = int(pos_indices[min(2, len(pos_indices) - 1)]) if len(pos_indices) > 0 else 0
        sample = ds[chosen_idx]

        xg = sample["global_img"].unsqueeze(0).to(device)
        xh = sample["highres_img"].unsqueeze(0).to(device)
        od_gt = sample["od_gt"].unsqueeze(0).to(device)

        with torch.no_grad():
            crop_img, stn_t = model.localizer(xg, xh, local_size=384, oracle_coords=od_gt * 0.85)
            f_glob_raw = model.global_encoder(xg)
            f_glob = model.ms_rnfl(f_glob_raw)
            m_q_logit = model.quality_branch(f_glob_raw)
            m_q = torch.sigmoid(m_q_logit)
            f_loc_raw = model.local_encoder(crop_img)
            f_loc_isnt = model.polar_isnt(f_loc_raw)
            f_loc_aligned = model.localizer.inverse_warp_local_to_global(f_loc_isnt, stn_t, f_glob.shape[-2], f_glob.shape[-1])
            f_fused, gate = model.qcaf(f_loc_aligned, f_glob, m_q)

        cls_w = model.cls_head.weight.detach().view(1, -1, 1, 1)
        cam_global = F.relu((f_fused * cls_w).sum(dim=1)).squeeze(0).cpu().numpy()
        cam_local = F.relu((f_loc_isnt * cls_w).sum(dim=1)).squeeze(0).cpu().numpy()
        mq_2d = m_q.squeeze().cpu().numpy()
        gate_2d = gate.mean(dim=1).squeeze(0).cpu().numpy()

        glob_rgb = denorm_img(sample["global_img"])
        crop_rgb = denorm_img(crop_img.squeeze(0))

        glob_box = glob_rgb.copy()
        cx, cy, s = stn_t[0].cpu().numpy()
        h, w, _ = glob_box.shape
        px, py = int((cx + 1.0) * 0.5 * w), int((cy + 1.0) * 0.5 * h)
        half_box = int(s * 0.5 * w)
        cv2.rectangle(
            glob_box,
            (max(5, px - half_box), max(5, py - half_box)),
            (min(w - 5, px + half_box), min(h - 5, py + half_box)),
            (0, 255, 128), 4
        )

        overlay_global = apply_colormap_overlay(glob_rgb, cam_global, alpha=0.45)
        overlay_local = apply_colormap_overlay(crop_rgb, cam_local, alpha=0.45)

        panels = [
            (glob_box, None),
            (crop_rgb, None),
            (mq_2d, "viridis"),
            (gate_2d, "magma"),
            (overlay_local, None),
            (overlay_global, None)
        ]

        for col_idx, (img_panel, cmap_name) in enumerate(panels):
            ax = axes[row_idx, col_idx]
            if cmap_name is None:
                ax.imshow(img_panel)
            else:
                im = ax.imshow(cv2.resize(img_panel, (256, 256), interpolation=cv2.INTER_CUBIC), cmap=cmap_name)
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontweight="bold", pad=8)
            if col_idx == 0:
                ax.set_ylabel(dname, fontweight="bold", fontsize=12)
            ax.set_xticks([])
            ax.set_yticks([])

    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "fig1_glaucogate_heatmaps_panel.png"), bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, "fig1_glaucogate_heatmaps_panel.pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"[SUCCESS] Saved Multi-Domain Heatmaps Panel to {out_dir}/fig1_glaucogate_heatmaps_panel.png", flush=True)

def generate_roc_and_degradation_plots(out_dir="./results/figures"):
    os.makedirs(out_dir, exist_ok=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    eval_pairs = [
        ("AIROGS In-Domain Test", "preds_airogs_test.csv"),
        ("PAPILA Zero-Shot External", "preds_papila_zeroshot.csv"),
        ("Chaksu Zero-Shot External", "preds_chaksu_zeroshot.csv")
    ]

    for idx, (title, fname) in enumerate(eval_pairs):
        ax = axes[idx]
        pro_path = os.path.join("./results/glaucogate_convnext_pro_seed42", fname)
        base_path = os.path.join("./results/baseline_convnext_base_seed42", fname)

        ax.axvspan(0.0, 0.10, color="#d4edda", alpha=0.45, label="90-100% Spec (pAUC Region)")

        if os.path.exists(pro_path):
            df_p = pd.read_csv(pro_path)
            fpr_p, tpr_p, _ = roc_curve(df_p["label"], df_p["logit"])
            auc_p = roc_auc_score(df_p["label"], df_p["logit"])
            pauc_p = roc_auc_score(df_p["label"], df_p["logit"], max_fpr=0.10)
            ax.plot(fpr_p, tpr_p, color="#d62728", lw=2.6, label=f"GlaucoGate (AUC={auc_p:.3f}, pAUC={pauc_p:.3f})")

        if os.path.exists(base_path):
            df_b = pd.read_csv(base_path)
            score_col = "logit" if "logit" in df_b.columns else "prob"
            fpr_b, tpr_b, _ = roc_curve(df_b["label"], df_b[score_col])
            auc_b = roc_auc_score(df_b["label"], df_b[score_col])
            pauc_b = roc_auc_score(df_b["label"], df_b[score_col], max_fpr=0.10)
            ax.plot(fpr_b, tpr_b, color="#1f77b4", lw=2.0, linestyle="--", label=f"ConvNeXt-Base (AUC={auc_b:.3f}, pAUC={pauc_b:.3f})")

        ax.plot([0, 1], [0, 1], "k:", alpha=0.5)
        ax.axvline(0.05, color="green", linestyle="-.", alpha=0.7, label="95% Specificity Operating Line")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("False Positive Rate (1 - Specificity)")
        ax.set_ylabel("True Positive Rate (Sensitivity)")
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.02])
        ax.grid(True, alpha=0.25)
        ax.legend(loc="lower right", frameon=True)

    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "fig2_roc_and_pauc_curves.png"), bbox_inches="tight")
    fig.savefig(os.path.join(out_dir, "fig2_roc_and_pauc_curves.pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"[SUCCESS] Saved ROC & pAUC Comparison Plots to {out_dir}/fig2_roc_and_pauc_curves.png", flush=True)

    deg_csv = "./results/degradation_study_table.csv"
    if os.path.exists(deg_csv):
        deg_df = pd.read_csv(deg_csv)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

        sub = deg_df[deg_df["Degradation"] == "disc_corruption"]
        if len(sub) == 0:
            sub = deg_df[deg_df["Degradation"] == deg_df["Degradation"].iloc[0]]

        for mod_name, color, marker in [("GlaucoGate_QCAF", "#d62728", "o"), ("NoQualityGate_Attn", "#1f77b4", "s")]:
            m_df = sub[sub["Model"] == mod_name].sort_values("Severity")
            if len(m_df) > 0:
                axes[0].plot(m_df["Severity"], m_df["Full_AUC"], marker=marker, lw=2.4, color=color, label=f"{mod_name} (Full AUC)")
                axes[0].plot(m_df["Severity"], m_df["Selective90Cov_Sens@95Sp"], marker=marker, lw=2.0, linestyle="--", color=color, alpha=0.8, label=f"{mod_name} (Sel@90%Cov Sens@95Sp)")
                axes[1].plot(m_df["Severity"], m_df["Gate_Inside_Corrupted_Region"], marker=marker, lw=2.4, color=color, label=f"{mod_name} Gate $G(\\Omega)$")

        axes[0].set_title("Diagnostic Robustness Under Optic Disc Corruption", fontweight="bold")
        axes[0].set_xlabel("Corruption Severity $\\sigma$")
        axes[0].set_ylabel("Metric Score")
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        axes[1].set_title("Q-CAF Gate Suppression Inside Corrupted Region $\\Omega$", fontweight="bold")
        axes[1].set_xlabel("Corruption Severity $\\sigma$")
        axes[1].set_ylabel("Mean Gate Activation Inside $\\Omega$")
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, "fig3_degradation_and_selective_risk.png"), bbox_inches="tight")
        fig.savefig(os.path.join(out_dir, "fig3_degradation_and_selective_risk.pdf"), bbox_inches="tight")
        plt.close(fig)
        print(f"[SUCCESS] Saved Degradation & Gate Suppression Plots to {out_dir}/fig3_degradation_and_selective_risk.png", flush=True)

if __name__ == "__main__":
    generate_multidomain_heatmaps("./checkpoints/glaucogate_convnext_pro_seed42_best.pt")
    generate_roc_and_degradation_plots()